In [ ]:
from langgraph.graph import StateGraph , START , END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel,Field

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

In [ ]:
class SentimentSchema(BaseModel):

  sentiment: Literal["positive" , "negative"] = Field(description='Sentiment of the review')

In [ ]:
class DiagnosisSchema(BaseModel):

  issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
  tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
  urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')


In [ ]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model_2 = model.with_structured_output(DiagnosisSchema)

In [ ]:
prompt = ' What is the sentiment of the following review- The Software is Good'
structured_model.invoke(prompt).sentiment

In [ ]:
class ReviewState(TypedDict):
  review: str
  sentiment: Literal["positive" , "negative"]
  diagnosis: dict
  respoanse: str

In [ ]:
def find_sentiment(state: ReviewState):
  prompt =f"""For the following review find out the sentiment \n {state["review"]}"""
  sentiment = structured_model.invoke(prompt).sentiment
  return {'sentiment': sentiment}

  def check_sentiment(state: ReviewState) -> Literal["positive_response" , "run_diagnosis"]:
    if state['sentiment'] == 'positive':
      return 'positive_response'
    else:
      return 'run_diagnosis'

  def positive_response(state: ReviewState):
    prompt = f 'Write a warm Thank you to this review \n {state['review']}'
    response = model.invoke(prompt)
    return {'response': response}

  def run_diagnosis(state: RevieState):
    prompt = f"""Diagnosis the issue in the following review \n {state['review']}"""
    diagnosis = structured_model_2.invoke(prompt)
    return {'diagnosis': diagnosis}


  def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
    response = model.invoke(prompt)
    return {'response': response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment': find_sentiment)
graph.add_node('positive_response' ,positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_respoane'negative_respoanse)

graph.add_edge(START, 'find_sentimnet')
graph.add_edge('find_sentiment' , 'check_sentiment')
graph.add_edge('run_diagnosis' , 'negative_response')
graph.add_edge('negative_response' , END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'review': "The product was really good"
}
workflow.invoke(initial_state)